# Stage 2 on Colab: steps 8–9 with other models / Этап 2 в Colab: шаги 8–9 на других моделях

**EN.** This notebook runs `notebooks/02_8_other_models.ipynb` (logistic regression on WoE, XGBoost, LightGBM, random forest) on Colab with the project code from the bundle. The bundle contains only the Y1 train sample and the Y1 validation rows; **test is not in it**.

Before running: (1) Runtime → Change runtime type → **CPU, High-RAM** (the random forest and LightGBM run on CPU; a GPU only speeds up XGBoost, set `XAI_XGB_DEVICE` below to `cuda` if you choose one); (2) upload `colab_bundle_stage2.zip` to Google Drive into the folder `AITU/xai_stage2`. Then Runtime → Run all. Fitted models are cached on Drive, so if the session stops, run all again and finished parts are skipped.

At the end download the Drive folder `AITU/xai_stage2/results` and give it to the project (`scripts/import_colab_stage2.py`).

**RU.** Ноутбук выполняет `notebooks/02_8_other_models.ipynb` (логистическая регрессия на WoE, XGBoost, LightGBM, случайный лес) в Colab с кодом проекта из архива. В архиве только выборка train и наблюдения validation для Y1; **test в нём нет**.

Перед запуском: (1) Runtime → Change runtime type → **CPU, High-RAM** (случайный лес и LightGBM считаются на процессоре; GPU ускоряет только XGBoost — если выбрали его, поставьте ниже `XAI_XGB_DEVICE = "cuda"`); (2) загрузите `colab_bundle_stage2.zip` на Google Drive в папку `AITU/xai_stage2`. Затем Runtime → Run all. Обученные модели сохраняются на Drive, поэтому если сессия прервётся, просто запустите всё заново — готовые части пропускаются.

В конце скачайте с Drive папку `AITU/xai_stage2/results` и передайте её в проект (`scripts/import_colab_stage2.py`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess, zipfile, glob
DRIVE = "/content/drive/MyDrive/AITU/xai_stage2"
ROOT = "/content/xai"
CACHE_DRIVE, RESULTS = f"{DRIVE}/models_cache", f"{DRIVE}/results"
os.makedirs(CACHE_DRIVE, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
if not os.path.exists(f"{ROOT}/src"):
    with zipfile.ZipFile(f"{DRIVE}/colab_bundle_stage2.zip") as z:
        z.extractall(ROOT)
# model predictions and step caches live on Drive, so a restarted session skips finished parts
cache_local = f"{ROOT}/data/interim/stage2/models"
if os.path.isdir(cache_local) and not os.path.islink(cache_local):
    shutil.rmtree(cache_local)
if not os.path.exists(cache_local):
    os.symlink(CACHE_DRIVE, cache_local)
for f in glob.glob(f"{CACHE_DRIVE}/step[89]_*.csv"):      # step caches of an earlier session
    shutil.copy(f, f"{ROOT}/data/interim/stage2/")
print(sorted(os.listdir(ROOT)))

In [ ]:
!pip -q install polars==2.0.0 xgboost==3.2.0 lightgbm==4.7.0 scikit-learn==1.9.1 pyyaml nbconvert nbclient ipykernel
!nproc; free -g | head -2

In [ ]:
XAI_XGB_DEVICE = "cpu"          # "cuda" on a GPU runtime
env = {**os.environ, "PYTHONPATH": f"{ROOT}/src", "XAI_XGB_DEVICE": XAI_XGB_DEVICE}
p = subprocess.Popen(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
                      "--ExecutePreprocessor.timeout=-1", "notebooks/02_8_other_models.ipynb"],
                     cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
    for f in glob.glob(f"{ROOT}/data/interim/stage2/step[89]_*.csv"):   # keep step caches on Drive as they appear
        shutil.copy(f, CACHE_DRIVE)
if p.wait() != 0:
    raise RuntimeError("02_8 failed: see the output above")

In [ ]:
shutil.copy(f"{ROOT}/notebooks/02_8_other_models.ipynb", RESULTS)
for f in glob.glob(f"{ROOT}/reports/feature_selection/other_models_*.csv") + glob.glob(f"{ROOT}/data/interim/stage2/step[89]_*.csv"):
    shutil.copy(f, RESULTS)
sorted(os.listdir(RESULTS))